In [ ]:
from core.data_sources import CLOBDataSource
from core.data_sources.hummingbot_database import HummingbotDatabase
from datetime import datetime
import sys
import os
import logging
import pandas as pd

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(root_path)


logging.getLogger("asyncio").setLevel(logging.CRITICAL)
logging.getLogger("pandas").setLevel(logging.CRITICAL)

db_names = [path for path in os.listdir(os.path.join(root_path, "data", "live_bot_databases", "brigado_server")) if path != ".gitignore"]
dbs = []
for db_name in db_names:
    if db_name.endswith(".sqlite"):
        db = HummingbotDatabase(db_name=db_name, server_name="brigado_server", root_path=root_path)
        dbs.append(db)

In [ ]:
stats = []

for db in dbs:
    if db.status["trade_fill"] == "Correct":
        trades_df = db.get_trade_fills()
        trades_df["date"] = pd.to_datetime(trades_df["timestamp"]).dt.strftime("%Y-%m-%d")
        first_row = trades_df.iloc[0]
        stats_dict = {
            "config_file_path": first_row["config_file_path"],
            "exchange": first_row["market"],
            "trading_pair": first_row["symbol"],
            "daily_quote_volume": trades_df.groupby("date")["amount"].sum().to_dict()
        }
        stats.append(stats_dict)

print(f"We found problems in the following databases: {[db.db_name for db in dbs if db.status["trade_fill"] != "Correct"]}")
stats_df = pd.DataFrame(stats)
stats_df

In [ ]:
df_expanded = (
    stats_df
    .set_index(["config_file_path", "exchange", "trading_pair"])
    ["daily_quote_volume"]
    .apply(pd.Series)
    .stack()
    .reset_index()
    .rename(columns={"level_3": "date", 0: "total_usdt_volume"})
)

df_expanded


In [ ]:
exchanges = list(df_expanded["exchange"].unique())
trading_pairs = list(df_expanded["trading_pair"].unique())
interval = "1d"
start_date = df_expanded["date"].min()
days = (datetime.now() - pd.to_datetime(start_date)).days

exchange_data = {exchange: [] for exchange in exchanges}
for exchange in exchanges:
    clob = CLOBDataSource()
    candles = await clob.get_candles_batch_last_days(connector_name=exchange,
                                                     trading_pairs=trading_pairs,
                                                     interval=interval,
                                                     days=days)
    for trading_pair in trading_pairs:
        candles_df = [candle.data for candle in candles if candle.trading_pair == trading_pair][0].copy()
        candles_df["date"] = pd.to_datetime(candles_df["timestamp"], unit="s").dt.strftime("%Y-%m-%d")
        exchange_data[exchange].append(
            {
                "candles_df": candles_df,
                "activity": df_expanded[df_expanded["exchange"] == exchange],
                "trading_pair": trading_pair,
            }
        )


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

first_target = 0.005
second_target = 0.01
exchange = "binance"
trading_pair = "USDT-BRL"

data = [data for data in exchange_data[exchange] if data["trading_pair"] == trading_pair][0]
fig = go.Figure()
candles_trace = go.Candlestick(x=data["candles_df"]["date"],
                               open=data["candles_df"]["open"],
                               high=data["candles_df"]["high"],
                               low=data["candles_df"]["low"],
                               close=data["candles_df"]["close"])
daily_volume_df = data["candles_df"].groupby("date")["volume"].sum().reset_index()
bot_daily_volume = data["activity"].groupby("date")["total_usdt_volume"].sum().reset_index()

In [ ]:
overall_volume_df = daily_volume_df.merge(bot_daily_volume, on="date", how="left")
overall_volume_df["target_0.01"] = overall_volume_df["volume"] * 0.01
overall_volume_df["market_participation"] = overall_volume_df["total_usdt_volume"] / overall_volume_df["volume"]

fig = go.Figure()

# Left axis (absolute volumes)
fig.add_trace(
    go.Bar(
        name="Daily 1% Target",
        x=overall_volume_df["date"],
        y=overall_volume_df["target_0.01"]
    )
)

fig.add_trace(
    go.Bar(
        name="Bot Daily Volume",
        x=overall_volume_df["date"],
        y=overall_volume_df["total_usdt_volume"],
        marker_color="lime"
    )
)

# Right axis (percentages)
fig.add_trace(
    go.Scatter(
        name="Total Market Share",
        x=overall_volume_df["date"],
        y=overall_volume_df["market_participation"],
        mode="lines+markers",
        yaxis="y2",  # 👈 put this trace on the right axis
        line=dict(color="white", width=2)
    )
)

# Configure axes
fig.update_layout(
    yaxis=dict(
        title="Volume (USDT)"
    ),
    yaxis2=dict(
        title="Market Share (%)",
        overlaying="y",       # share same x
        side="right",
        tickformat=".2%"      # format as percentage
    ),
    barmode="group",
    height=800
)

# Print summary
total_bot_volume = overall_volume_df["total_usdt_volume"].sum()
print(f"Overall Bot Volume (USDT): {total_bot_volume}")

fig.show()


In [ ]:
df_expanded